# XModalix3D three-modality full-data alignment test

This notebook trains **XModalix3D** on the full paired NCCT, CBF and CBV datasets for 250 epochs and evaluates latent-space alignment.

The notebook is designed to be executed non-interactively on the HPC. Paths are taken from environment variables set by the accompanying SLURM script. No pipeline/model pickle is written.

**Important:** because `requires_paired=True`, the training set contains the intersection of sample IDs that are available in all three modalities.

## Environment and configuration

In [ ]:
import os
import sys
import inspect
from pathlib import Path

import torch
import autoencodix as acx

from autoencodix.configs import DataCase, DataConfig, DataInfo
from autoencodix.configs.xmodalix3d_config import XModalix3DConfig
from autoencodix.utils._losses import XModalLoss

DATA_ROOT = Path(
    os.environ.get(
        "DATA_ROOT",
        "/data/horse/ws/baeuchl-imagix3d/data/stroke_data",
    )
)

ncctpath = os.environ.get(
    "NCCT_PATH",
    str(DATA_ROOT / "ncct_flat"),
)
ncctannopath = os.environ.get(
    "NCCT_ANNO",
    str(DATA_ROOT / "ncct_anno_dst.csv"),
)
cbfpath = os.environ.get(
    "CBF_PATH",
    str(DATA_ROOT / "cbf_flat"),
)
cbfannopath = os.environ.get(
    "CBF_ANNO",
    str(DATA_ROOT / "cbf_anno_dst.csv"),
)
cbvpath = os.environ.get(
    "CBV_PATH",
    str(DATA_ROOT / "cbv_flat"),
)
cbvannopath = os.environ.get(
    "CBV_ANNO",
    str(DATA_ROOT / "cbv_anno_dst.csv"),
)

paths_to_check = {
    "NCCT volumes": Path(ncctpath),
    "NCCT annotation": Path(ncctannopath),
    "CBF volumes": Path(cbfpath),
    "CBF annotation": Path(cbfannopath),
    "CBV volumes": Path(cbvpath),
    "CBV annotation": Path(cbvannopath),
}

for name, path in paths_to_check.items():
    print(f"{name}: {path}")
    assert path.exists(), f"Missing input: {name}: {path}"

print()
print("Python:", sys.executable)
print("AUTOENCODIX:", acx.__file__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

assert torch.cuda.is_available(), "This HPC notebook is configured for CUDA."

print("\nActive XModal adversarial-loss implementation:\n")
print(inspect.getsource(XModalLoss._calc_adversial_loss))

config = XModalix3DConfig(
    data_case=DataCase.IMG_TO_IMG,

    # 3D handling
    img_path_col="filepath",
    spatial_shape_policy="crop_or_pad_to_shape",
    target_shape_3d=(160, 192, 160),
    volume_scaling_strategy="per_volume",
    normalize_nonzero_only=True,

    # Full-data alignment run
    epochs=250,
    checkpoint_interval=5,
    batch_size=5,
    latent_dim=64,
    hidden_dim=16,
    pretrain_epochs=0,
    delta_class=0,

    reconstruction_loss="mse",
    loss_reduction="mean",
    scaling="NONE",

    # Reproducibility
    reproducible=True,
    global_seed=1,

    # XModalix alignment weights
    gamma=30,
    delta_pair=5,

    train_normalization="group",
    train_norm_groupsize=8,

    clamp_logvar=True,
    keep_mu_positive=False,

    requires_paired=True,
    device="cuda",
    n_gpus=1,

    data_config=DataConfig(
        data_info={
            "NCCT": DataInfo(
                file_path=ncctpath,
                extra_anno_file=ncctannopath,
                data_type="IMG",
                scaling="NONE",
                translate_direction="from",
                pretrain_epochs=0,
            ),
            "CBF": DataInfo(
                file_path=cbfpath,
                extra_anno_file=cbfannopath,
                data_type="IMG",
                scaling="NONE",
                translate_direction="to",
                pretrain_epochs=0,
            ),
            "CBV": DataInfo(
                file_path=cbvpath,
                extra_anno_file=cbvannopath,
                data_type="IMG",
                scaling="NONE",
                translate_direction=None,
                pretrain_epochs=0,
            ),
            "ANNO": DataInfo(
                file_path=ncctannopath,
                data_type="ANNOTATION",
            ),
        }
    ),
)

print("\nKey run configuration:")
for key in [
    "epochs",
    "checkpoint_interval",
    "batch_size",
    "latent_dim",
    "hidden_dim",
    "gamma",
    "delta_pair",
    "delta_class",
    "reproducible",
    "global_seed",
    "device",
]:
    print(f"{key}: {getattr(config, key)}")


## Memory-saving checkpoint capture

The generic XModal trainer normally stores full reconstructed 3D volumes at every checkpoint. With full-resolution volumes and a checkpoint every 5 epochs, retaining those reconstructions would consume very large amounts of RAM.

For this alignment experiment we only need the latent means (`mu`) and sample IDs at each checkpoint. The following local monkey patch changes **result capture only**; it does not alter the model forward pass, losses, gradients, optimizer, or training objective. The prediction step can still add the requested translation/reference reconstructions to the otherwise empty reconstruction dictionary.

In [ ]:
import numpy as np

from autoencodix.trainers._xmodal_trainer import XModalTrainer


def _capture_alignment_dynamics(self, batch_data):
    captured_data = {
        "latentspaces": {},
        "reconstructions": {},
        "mus": {},
        "sigmas": {},
        "sample_ids": {},
    }

    for mod_name, dynamics in self._modality_dynamics.items():
        sample_ids = batch_data[mod_name].get("sample_ids")

        if sample_ids is not None:
            captured_data["sample_ids"][mod_name] = np.asarray(sample_ids)

        model_output = dynamics["mp"]

        if model_output.latent_mean is not None:
            captured_data["mus"][mod_name] = model_output.latent_mean.detach()

    return captured_data


XModalTrainer._capture_dynamics = _capture_alignment_dynamics

print(
    "Patched XModalTrainer._capture_dynamics: "
    "checkpoint capture stores only mu + sample IDs."
)


## Preprocessing and integration checks

In [ ]:
from autoencodix.base._base_dataset import DataSetTypes
from autoencodix.data._xmodal3d_preprocessor import XModal3DPreprocessor
from autoencodix.modeling._volumevae_architecture import VolumeVAEArchitecture

xmodal3d = acx.XModalix3D(config=config)

xmodal3d.preprocess()

expected_modalities = {
    "img.NCCT",
    "img.CBF",
    "img.CBV",
}

for split_name in ["train", "valid", "test"]:
    split = getattr(
        xmodal3d._datasets,
        split_name,
    )

    print(f"\n--- {split_name} ---")
    print("modalities:", split.datasets.keys())
    print("fully paired:", split.is_fully_paired)

    assert set(split.datasets.keys()) == expected_modalities
    assert split.is_fully_paired

    id_sets = []

    for modality, ds in split.datasets.items():
        print(
            modality,
            "n =", len(ds),
            "shape =", tuple(ds[0][1].shape),
        )

        id_sets.append(set(ds.sample_ids))

    assert all(
        ids == id_sets[0]
        for ids in id_sets[1:]
    )

print("\nPreprocessing/pairing checks passed.")


## Inspect one multimodal batch

In [ ]:
from torch.utils.data import DataLoader
from autoencodix.data._multimodal_dataset import (
    create_multimodal_collate_fn,
)

train = xmodal3d._datasets.train

loader = DataLoader(
    train,
    batch_size=2,
    shuffle=False,
    collate_fn=create_multimodal_collate_fn(train),
)

batch = next(iter(loader))

assert set(batch.keys()) == expected_modalities

for modality, contents in batch.items():
    print(
        modality,
        contents["data"].shape,
        contents["sample_ids"],
    )


## Train XModalix3D

In [ ]:
xmodal3d.fit()

## Predict NCCT → CBF on the held-out test split

In [ ]:
xmodal3d.predict(
    from_key="NCCT",
    to_key="CBF",
)

## Alignment helper functions

In [ ]:
import numpy as np
import pandas as pd

from itertools import combinations

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
)


def get_aligned_pair(
    latent_dict,
    id_dict,
    modality_a,
    modality_b,
):
    ids_a = np.asarray(id_dict[modality_a]).astype(str)
    ids_b = np.asarray(id_dict[modality_b]).astype(str)

    z_a = np.asarray(latent_dict[modality_a])
    z_b = np.asarray(latent_dict[modality_b])

    idx_a = {sid: i for i, sid in enumerate(ids_a)}
    idx_b = {sid: i for i, sid in enumerate(ids_b)}

    common_ids = sorted(set(ids_a) & set(ids_b))

    A = np.stack([z_a[idx_a[sid]] for sid in common_ids])
    B = np.stack([z_b[idx_b[sid]] for sid in common_ids])

    return common_ids, A, B


def pairwise_alignment_report_exact(
    latent_dict,
    id_dict,
):
    """Compare matched cross-modal distances with all unmatched patient pairs."""
    modalities = sorted(latent_dict.keys())
    rows = []

    for a, b in combinations(modalities, 2):
        common_ids, A, B = get_aligned_pair(
            latent_dict,
            id_dict,
            a,
            b,
        )

        matched_distances = np.linalg.norm(
            A - B,
            axis=1,
        )
        matched_mean = matched_distances.mean()

        distance_matrix = np.linalg.norm(
            A[:, None, :] - B[None, :, :],
            axis=2,
        )

        off_diagonal_mask = ~np.eye(
            len(common_ids),
            dtype=bool,
        )

        unmatched_distances = distance_matrix[
            off_diagonal_mask
        ]
        unmatched_mean = unmatched_distances.mean()

        rows.append({
            "modality_a": a,
            "modality_b": b,
            "n_pairs": len(common_ids),
            "matched_distance": matched_mean,
            "unmatched_distance": unmatched_mean,
            "matched/unmatched_ratio": (
                matched_mean / unmatched_mean
                if unmatched_mean > 0
                else np.nan
            ),
        })

    return pd.DataFrame(rows)


def build_modality_probe_data(
    latent_dict,
    id_dict,
):
    X = []
    y = []
    groups = []

    for modality, z in latent_dict.items():
        z = np.asarray(z)
        ids = np.asarray(id_dict[modality]).astype(str)

        for row, sid in zip(z, ids):
            X.append(row)
            y.append(modality)
            groups.append(sid)

    return (
        np.asarray(X),
        np.asarray(y),
        np.asarray(groups),
    )


def modality_probe_score(
    latent_dict,
    id_dict,
):
    X, y, groups = build_modality_probe_data(
        latent_dict,
        id_dict,
    )

    modalities = sorted(np.unique(y))
    n_groups = len(np.unique(groups))

    cv = GroupKFold(
        n_splits=min(5, n_groups)
    )

    probe = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
        ),
    )

    pred = cross_val_predict(
        probe,
        X,
        y,
        cv=cv,
        params={"groups": groups},
    )

    score = balanced_accuracy_score(
        y,
        pred,
    )
    chance = 1.0 / len(modalities)

    cm = confusion_matrix(
        y,
        pred,
        labels=modalities,
    )

    cm_df = pd.DataFrame(
        cm,
        index=modalities,
        columns=modalities,
    )

    return score, chance, cm_df


## Final training and test alignment

In [ ]:
available_epochs = sorted(
    epoch
    for epoch, split_data
    in xmodal3d.result.mus._data.items()
    if (
        split_data is not None
        and "train" in split_data
        and split_data["train"] is not None
    )
)

assert available_epochs, "No stored training mu checkpoints were found."

final_epoch = max(available_epochs)

print("Final stored internal epoch:", final_epoch)
print("Human-readable epoch:", final_epoch + 1)

train_mu = xmodal3d.result.mus.get(
    epoch=final_epoch,
    split="train",
)
train_ids = xmodal3d.result.sample_ids.get(
    epoch=final_epoch,
    split="train",
)

test_mu = xmodal3d.result.mus.get(
    epoch=-1,
    split="test",
)
test_ids = xmodal3d.result.sample_ids.get(
    epoch=-1,
    split="test",
)

print("\nTRAIN pairwise alignment:")
display(
    pairwise_alignment_report_exact(
        train_mu,
        train_ids,
    )
)

print("\nTEST pairwise alignment:")
display(
    pairwise_alignment_report_exact(
        test_mu,
        test_ids,
    )
)


## Final modality-probe diagnostics

In [ ]:
train_probe, chance, train_cm = modality_probe_score(
    train_mu,
    train_ids,
)

test_probe, test_chance, test_cm = modality_probe_score(
    test_mu,
    test_ids,
)

print("TRAIN balanced modality accuracy:", train_probe)
print("TRAIN chance:", chance)
display(train_cm)

print("\nTEST balanced modality accuracy:", test_probe)
print("TEST chance:", test_chance)
display(test_cm)


## Check against trivial latent collapse

In [ ]:
for split_name, mu_dict in [
    ("train", train_mu),
    ("test", test_mu),
]:
    print(f"\n{split_name.upper()}")
    for modality, z in mu_dict.items():
        z = np.asarray(z)
        print(
            modality,
            "mean latent std =",
            np.std(z, axis=0).mean(),
        )


## Alignment across training checkpoints

In [ ]:
print(
    "Stored checkpoint epochs:",
    available_epochs,
)

print(
    "Human-readable training epochs:",
    [epoch + 1 for epoch in available_epochs],
)

probe_rows = []
pair_rows = []

for epoch in available_epochs:
    checkpoint_mu = xmodal3d.result.mus.get(
        epoch=epoch,
        split="train",
    )

    checkpoint_ids = xmodal3d.result.sample_ids.get(
        epoch=epoch,
        split="train",
    )

    probe_score, chance, _ = modality_probe_score(
        checkpoint_mu,
        checkpoint_ids,
    )

    probe_rows.append({
        "epoch_internal": epoch,
        "epoch": epoch + 1,
        "probe_balanced_accuracy": probe_score,
        "chance": chance,
    })

    pair_report = pairwise_alignment_report_exact(
        checkpoint_mu,
        checkpoint_ids,
    )

    pair_report["epoch_internal"] = epoch
    pair_report["epoch"] = epoch + 1

    pair_rows.append(pair_report)

probe_over_time = pd.DataFrame(probe_rows)
pairwise_over_time = pd.concat(
    pair_rows,
    ignore_index=True,
)

display(probe_over_time)

display(
    pairwise_over_time[
        [
            "epoch",
            "modality_a",
            "modality_b",
            "matched_distance",
            "unmatched_distance",
            "matched/unmatched_ratio",
        ]
    ]
)


## Plot alignment evolution

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(
    probe_over_time["epoch"],
    probe_over_time["probe_balanced_accuracy"],
    marker="o",
    label="Modality probe",
)

plt.axhline(
    y=probe_over_time["chance"].iloc[0],
    linestyle="--",
    label="Chance",
)

plt.xlabel("Training epoch")
plt.ylabel("Balanced modality accuracy")
plt.title("Modality separability across training")
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))

for (
    modality_a,
    modality_b,
), group in pairwise_over_time.groupby(
    ["modality_a", "modality_b"]
):
    plt.plot(
        group["epoch"],
        group["matched/unmatched_ratio"],
        marker="o",
        label=f"{modality_a} ↔ {modality_b}",
    )

plt.axhline(
    y=1.0,
    linestyle="--",
    label="No patient-specific alignment",
)

plt.xlabel("Training epoch")
plt.ylabel("Matched / unmatched distance")
plt.title("Patient-specific latent alignment across training")
plt.legend()
plt.show()


## Completion

In [ ]:
print("XModalix3D full-data three-modality alignment notebook completed successfully.")
print("No pipeline/model pickle was saved by this notebook.")